# pacotes

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from datetime import datetime

from src.agro_predictor.data_loaders.tools_db import (salva_clima, obter_dados_clima, 
                       etl_fetch_market_data, db_save_market_data,
                       get_db_connection
                      )
from src.agro_predictor.data_loaders.tools_yf import ingest_exchange_rate, ingest_commodities
from src.agro_predictor.utils.tools_cy import load_json_data


# Configuração

In [2]:
# Configurações da região configurad no .env | ex: Sorriso - MT

LATITUDE_SORRISO = os.getenv('LATITUDE_SORRISO')
LONGITUDE_SORRISO = os.getenv('LONGITUDE_SORRISO')
NOME_REGIAO = os.getenv('NOME_REGIAO')
START_DATE = os.getenv('START_DATE')
END_DATE = datetime.today().strftime('%Y-%m-%d')
TICKER_SOJA = os.getenv("TICKER_SOJA")
MAP_COMMODITIES_JSON = os.getenv('MAP_COMMODITIES')

# Forçar encoding via Variável de Ambiente
os.environ["PGCLIENTENCODING"] = "utf-8"

# cria conexao com base
engine = get_db_connection()

# extração - open clima

In [3]:
# Obter dados clima

df_weather = obter_dados_clima(
    LATITUDE_SORRISO, 
    LONGITUDE_SORRISO, 
    START_DATE, 
    END_DATE, 
    NOME_REGIAO
)


🌦️ Buscando dados climáticos para: Sorriso_MT (2023-01-01 a 2026-04-01)...
✅ Download concluído: 1187 registros obtidos.


In [4]:
df_weather

,record_date,temp_max,temp_min,precipitation_mm,soil_moisture,region_name,latitude,longitude
0,2023-01-01,29.6,20.8,1.0,0.474,Sorriso_MT,-12.5425,-55.7214
1,2023-01-02,28.5,21.9,10.2,0.477,Sorriso_MT,-12.5425,-55.7214
2,2023-01-03,29.3,21.5,2.1,0.481,Sorriso_MT,-12.5425,-55.7214
3,2023-01-04,29.2,21.3,17.1,0.491,Sorriso_MT,-12.5425,-55.7214
4,2023-01-05,28.2,22.1,4.8,0.501,Sorriso_MT,-12.5425,-55.7214
...,...,...,...,...,...,...,...,...
1182,2026-03-28,29.4,22.4,7.4,0.488,Sorriso_MT,-12.5425,-55.7214
1183,2026-03-29,30.9,21.4,0.3,0.465,Sorriso_MT,-12.5425,-55.7214
1184,2026-03-30,30.6,21.9,0.4,0.424,Sorriso_MT,-12.5425,-55.7214
1185,2026-03-31,30.1,21.5,0.3,0.387,Sorriso_MT,-12.5425,-55.7214


In [5]:
# Gravar clima

salva_clima(df_weather, engine)


🔄 Processando 1187 registros para 1 regiões entre 2023-01-01 e 2026-04-01...
💾 Salvando 3 novos registros...
🚀 Sucesso! 3 registros inseridos.


# extração - yf

In [6]:
# Executa ETL

df_clean = etl_fetch_market_data(TICKER_SOJA, START_DATE)


🔄 [ETL] Buscando dados para ZS=F a partir de 2023-01-01...
✅ [ETL] Transformação concluída: 815 registros preparados.


In [7]:
# Executa Gravação

db_save_market_data(df_clean, engine, 'market_data', 'finance', )


🚀 [DB] Sucesso! 3 novos registros inseridos em finance.market_data.


# inserir preço | commodieties

In [8]:
# Inserindo dados - preço

ingest_exchange_rate(START_DATE, engine)


Baixando dados reais do Dólar (BRL=X)...
Última data no banco: 2026-03-29
Inserindo 2 NOVOS registros de câmbio no banco...
63: Câmbio atualizado com sucesso!


In [9]:
# Carregar regra

commodities_map = load_json_data(MAP_COMMODITIES_JSON)

if commodities_map: 
    for db_ticker, yahoo_ticker in commodities_map.items():
        ingest_commodities(yahoo_ticker, db_ticker, START_DATE, engine)
else:
    print("O mapeamento não pôde ser carregado. O processo foi interrompido.")


14: |../init_scripts/map_commodities.json|
--- Processando: SOJA (ZS=F) ---
Inserindo 815 registros de SOJA...
✔ Sucesso: SOJA inserido/atualizado.
--- Processando: CAFE (KC=F) ---
Inserindo 817 registros de CAFE...
✔ Sucesso: CAFE inserido/atualizado.
--- Processando: AVEIA (ZO=F) ---
Inserindo 815 registros de AVEIA...
✔ Sucesso: AVEIA inserido/atualizado.
